In [1]:
# Cell 1 — Setup: discover data paths and summarise what's on disk
from pathlib import Path
import glob, json, time

import pandas as pd

BASE_DIR   = Path.cwd().resolve()
ROOT       = BASE_DIR.parent if BASE_DIR.name.lower() == "spark" else BASE_DIR
DATA_DIR   = ROOT / "data"
STATE_FILE = DATA_DIR / ".push_state.json"

TABLES = {
    "trades":    DATA_DIR / "bronze" / "btc_trades",
    "klines":    DATA_DIR / "bronze" / "btc_klines",
    "ticker":    DATA_DIR / "bronze" / "btc_ticker",
    "orderbook": DATA_DIR / "bronze" / "btc_orderbook",
    "silver":    DATA_DIR / "silver" / "btc_aggregates",
}

def all_parquet(path: Path) -> list[Path]:
    return sorted(path.glob("*.parquet"))

def load_all(path: Path) -> pd.DataFrame:
    files = all_parquet(path)
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

# Summary table
rows = []
for name, path in TABLES.items():
    files = all_parquet(path)
    latest_mtime = max((f.stat().st_mtime for f in files), default=None)
    rows.append({
        "table":       name,
        "files":       len(files),
        "latest_file": pd.Timestamp(latest_mtime, unit="s", tz="UTC") if latest_mtime else pd.NaT,
    })
summary = pd.DataFrame(rows)

# Airflow push-state cursor
if STATE_FILE.exists():
    push_state = json.loads(STATE_FILE.read_text())
    print("Airflow cursor (last ingested_at pushed):")
    for k, v in push_state.items():
        print(f"  {k}: {v}")
    print()
else:
    print("No .push_state.json found (Airflow hasn't run yet).\n")

print("Files on disk:")
display(summary)


Airflow cursor (last ingested_at pushed):
  btc_trades: 2026-04-07T22:10:23.697435+00:00
  btc_ticker: 2026-04-07T22:10:19.146764+00:00
  btc_orderbook: 2026-04-07T22:10:22.942165+00:00
  btc_klines: 2026-04-07T22:10:02.169353+00:00

Files on disk:


,table,files,latest_file
0,trades,164,2026-04-07 22:16:19.261240482+00:00
1,klines,26,2026-04-07 22:10:08.184030771+00:00
2,ticker,52,2026-04-07 22:16:29.941487312+00:00
3,orderbook,154,2026-04-07 22:16:25.711660147+00:00
4,silver,42,2026-04-07 22:16:23.282246113+00:00


In [2]:
# Cell 2 — Bronze trades: latest rows + basic price/volume stats
try:
    df = load_all(TABLES["trades"])
    df["event_time"] = pd.to_datetime(df["event_time"], utc=True, errors="coerce")
    df = df.sort_values("event_time", ascending=False)

    print(f"Total trade records: {len(df):,}")
    print(f"Time range: {df['event_time'].min()}  →  {df['event_time'].max()}")
    print(f"Null price/volume: price={df['price'].isna().sum()}  volume={df['volume'].isna().sum()}")
    print(f"price > 0: {(df['price'] > 0).all()}   volume > 0: {(df['volume'] > 0).all()}")
    print()
    print("Latest 20 rows:")
    display(df.head(20))
    print("\nPrice / volume distribution:")
    display(df[["price", "volume"]].describe())
except Exception as e:
    print(f"Trades not available: {e}")


Total trade records: 295,450
Time range: 2026-04-06 20:44:47.830000+00:00  →  2026-04-07 22:10:21.570000+00:00
Null price/volume: price=0  volume=0
price > 0: True   volume > 0: True

Latest 20 rows:


,symbol,price,volume,event_time,ingested_at
295449,BTCUSDT,70209.99,0.00637,2026-04-07 22:10:21.570000+00:00,2026-04-07 22:10:23.697435+00:00
295434,BTCUSDT,70207.05,0.00192,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.626497+00:00
295448,BTCUSDT,70208.48,0.00008,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.631810+00:00
295447,BTCUSDT,70208.33,0.00114,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.631794+00:00
295446,BTCUSDT,70208.00,0.00040,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.631774+00:00
295445,BTCUSDT,70208.00,0.00040,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.631743+00:00
295444,BTCUSDT,70207.72,0.00008,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.630810+00:00
295443,BTCUSDT,70207.06,0.00008,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.630069+00:00
295442,BTCUSDT,70207.06,0.00008,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.630053+00:00
295441,BTCUSDT,70207.05,0.00008,2026-04-07 22:10:21.492000+00:00,2026-04-07 22:10:23.630036+00:00



Price / volume distribution:


,price,volume
count,295450.000000,295450.000000
mean,69924.786876,0.005255
std,280.887138,0.076817
min,69316.820000,0.000010
25%,69741.272500,0.000080
50%,69905.300000,0.000080
75%,70079.010000,0.000150
max,70593.780000,14.317640


In [3]:
# Cell 3 — Silver aggregates: 1-min windows
try:
    df = load_all(TABLES["silver"])
    df["window_start"] = pd.to_datetime(df["window_start"], utc=True, errors="coerce")
    df["window_end"]   = pd.to_datetime(df["window_end"],   utc=True, errors="coerce")
    df = df.sort_values("window_start", ascending=False)

    print(f"Total silver rows: {len(df):,}")
    print(f"Window range: {df['window_start'].min()}  →  {df['window_start'].max()}")
    print()
    print("Latest 15 aggregation windows:")
    display(df.head(15))
    print("\nAvg price / total volume distribution:")
    display(df[["avg_price", "total_volume"]].describe())
except Exception as e:
    print(f"Silver data not available: {e}")


Total silver rows: 80
Window range: 2026-04-06 20:44:00+00:00  →  2026-04-07 22:10:00+00:00

Latest 15 aggregation windows:


,window_start,window_end,symbol,avg_price,total_volume
79,2026-04-07 22:10:00+00:00,2026-04-07 22:11:00+00:00,BTCUSDT,70183.711358,15.49770
78,2026-04-07 22:09:00+00:00,2026-04-07 22:10:00+00:00,BTCUSDT,70264.155721,34.82960
77,2026-04-07 22:08:00+00:00,2026-04-07 22:09:00+00:00,BTCUSDT,70361.902868,28.36776
76,2026-04-07 22:07:00+00:00,2026-04-07 22:08:00+00:00,BTCUSDT,70428.184782,51.87526
75,2026-04-07 22:06:00+00:00,2026-04-07 22:07:00+00:00,BTCUSDT,70472.740933,182.02115
74,2026-04-07 22:05:00+00:00,2026-04-07 22:06:00+00:00,BTCUSDT,70214.092106,113.00784
73,2026-04-07 22:04:00+00:00,2026-04-07 22:05:00+00:00,BTCUSDT,70077.076093,27.35821
72,2026-04-07 22:03:00+00:00,2026-04-07 22:04:00+00:00,BTCUSDT,70052.310901,9.82451
71,2026-04-07 22:02:00+00:00,2026-04-07 22:03:00+00:00,BTCUSDT,70042.661938,11.84630
70,2026-04-07 22:01:00+00:00,2026-04-07 22:02:00+00:00,BTCUSDT,70096.819669,46.34186



Avg price / total volume distribution:


,avg_price,total_volume
count,80.000000,80.000000
mean,69817.146201,19.218902
std,249.198393,27.257789
min,69323.700390,0.001810
25%,69663.111041,5.233747
50%,69850.959884,11.030895
75%,69937.814729,20.824365
max,70472.740933,182.021150


In [ ]:
# Cell 4 — Bronze klines: 1-min OHLCV candles
try:
    df = load_all(TABLES["klines"])
    df["open_time"]  = pd.to_datetime(df["open_time"],  utc=True, errors="coerce")
    df["close_time"] = pd.to_datetime(df["close_time"], utc=True, errors="coerce")
    df = df.sort_values("open_time", ascending=False)

    print(f"Total kline records: {len(df):,}")
    print(f"Time range: {df['open_time'].min()}  →  {df['open_time'].max()}")
    print()
    display(df[["open_time", "close_time", "open_price", "high_price", "low_price",
                "close_price", "volume", "trade_count"]].head(20))
    print("\nOHLCV distribution:")
    display(df[["open_price", "high_price", "low_price", "close_price", "volume"]].describe())
except Exception as e:
    print(f"Klines not available: {e}")


,timestamp,operation,version
0,1775343470671,STREAMING UPDATE,10
1,1775343460775,STREAMING UPDATE,9
2,1775343450752,STREAMING UPDATE,8
3,1775343440790,STREAMING UPDATE,7
4,1775343430846,STREAMING UPDATE,6
5,1775343420777,STREAMING UPDATE,5
6,1775343410779,STREAMING UPDATE,4
7,1775343403167,STREAMING UPDATE,3
8,1775343401108,STREAMING UPDATE,2
9,1775343390383,STREAMING UPDATE,1


In [ ]:
# Cell 5 — Bronze ticker: 24h rolling stats
try:
    df = load_all(TABLES["ticker"])
    df["event_time"] = pd.to_datetime(df["event_time"], utc=True, errors="coerce")
    df = df.sort_values("event_time", ascending=False)

    print(f"Total ticker snapshots: {len(df):,}")
    print(f"Time range: {df['event_time'].min()}  →  {df['event_time'].max()}")
    print()
    display(df[["event_time", "price_change", "price_change_pct",
                "weighted_avg_price", "high_price", "low_price",
                "volume_24h", "trade_count_24h"]].head(15))
except Exception as e:
    print(f"Ticker data not available: {e}")


,symbol,count
0,BTCUSDT,1003


In [ ]:
# Cell 6 — Bronze orderbook: bid/ask spread
try:
    df = load_all(TABLES["orderbook"])
    df["event_time"] = pd.to_datetime(df["event_time"], utc=True, errors="coerce")
    df = df.sort_values("event_time", ascending=False)

    print(f"Total orderbook snapshots: {len(df):,}")
    print(f"Time range: {df['event_time'].min()}  →  {df['event_time'].max()}")
    print()
    display(df[["event_time", "best_bid_price", "best_ask_price",
                "spread", "spread_pct", "best_bid_qty", "best_ask_qty"]].head(15))
    print("\nSpread distribution:")
    display(df[["spread", "spread_pct"]].describe())
except Exception as e:
    print(f"Orderbook data not available: {e}")


In [ ]:
# Cell 7 — Pipeline health: file age, ingestion gaps, null checks
now = pd.Timestamp.now(tz="UTC")
print(f"Health check at {now}\n")

issues = []
for name, path in TABLES.items():
    files = all_parquet(path)
    if not files:
        issues.append(f"  [{name}] NO FILES FOUND")
        print(f"  [{name}]    0 files  [NO FILES]")
        continue
    latest_mtime = max(f.stat().st_mtime for f in files)
    age_min = (time.time() - latest_mtime) / 60
    status = "OK" if age_min < 10 else "STALE"
    line = f"  [{name:>9}] {len(files):>4} files  last write {age_min:6.1f} min ago  [{status}]"
    print(line)
    if status == "STALE":
        issues.append(line)

# Ingestion gap check on trades — flag windows > 2 min with no trade
try:
    trades = load_all(TABLES["trades"])
    trades["event_time"] = pd.to_datetime(trades["event_time"], utc=True, errors="coerce")
    trades = trades.sort_values("event_time").dropna(subset=["event_time"])
    gaps = trades["event_time"].diff().dt.total_seconds().dropna()
    big_gaps = gaps[gaps > 120]
    print(f"\nTrade gaps > 2 min: {len(big_gaps)}")
    if not big_gaps.empty:
        gap_df = trades.loc[big_gaps.index, "event_time"].rename("gap_at").to_frame()
        gap_df["gap_seconds"] = big_gaps.values
        display(gap_df.sort_values("gap_seconds", ascending=False).head(5))
except Exception as e:
    print(f"\nCould not run gap analysis: {e}")

print("\n" + ("No issues detected." if not issues else "WARNINGS:\n" + "\n".join(issues)))
